# 05. Avaliar atribuição — mãe + endereço

Depois do [`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb)
(modelo `p ≥ 0,90` + resgate R1/R2/R3).

Avalia notas, qualidade, quem ficou de fora, amostra por faixa/regra e
números com lista ouro.

Entradas:
- `splink_predictions_mae_endereco.parquet` ([`02d`](02d_aplicar_splink_mae_endereco.ipynb))
- `splink_atribuicao_mae_endereco.parquet` ([`04`](04_atribuir_mae_endereco.ipynb))


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    LISTA_OURO_ARQUIVO,
    MAX_CENSOS_POR_CPF,
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    TABELA_CENSO_LIMPA,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    cpf_norm_sql,
    drop_splink_temp_tables,
    get_connection,
    lista_ouro_colunas,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO_MODELO = 0.90
PISO_RESGATE = 0.50
PISO_BAIXO = 0.25
TETO = MAX_CENSOS_POR_CPF

print_paths()
require_input(
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    label='SPLINK_PREDICTIONS_MAE_ENDERECO (rode o 02d antes)',
)
require_input(
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    label='SPLINK_ATRIBUICAO_MAE_ENDERECO (rode o 04_atribuir_mae_endereco antes)',
)
require_input(LISTA_OURO_ARQUIVO, label='LISTA_OURO')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA, TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print(f'Faixas 04: modelo>={PISO_MODELO} | resgate>={PISO_RESGATE} | baixo>={PISO_BAIXO} | teto={TETO}')


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

In [2]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')

con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS_MAE_ENDERECO}')
''')

# Nome distinto da TABLE atribuicao do 04 no mesmo DuckDB.
for _nome in ('atribuicao_aval', 'atribuicao_2', 'atribuicao'):
    _t = con.execute(f'''
    SELECT table_type
    FROM information_schema.tables
    WHERE table_schema = 'main' AND table_name = '{_nome}'
    ''').fetchone()
    if _t:
        _k = 'VIEW' if _t[0].upper() == 'VIEW' else 'TABLE'
        con.execute(f'DROP {_k} IF EXISTS {_nome}')

cols_atr = [
    r[0]
    for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{SPLINK_ATRIBUICAO_MAE_ENDERECO}') LIMIT 0"
    ).fetchall()
]
tem_faixa = 'faixa' in cols_atr
tem_regra = 'regra' in cols_atr
col_faixa = 'faixa' if tem_faixa else 'CAST(NULL AS VARCHAR) AS faixa'
col_regra = 'regra' if tem_regra else 'CAST(NULL AS VARCHAR) AS regra'

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_aval AS
SELECT
    unique_id_censo,
    unique_id_cpf,
    match_probability,
    {col_faixa},
    {col_regra}
FROM read_parquet('{SPLINK_ATRIBUICAO_MAE_ENDERECO}')
''')

n_app = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''').fetchone()[0]
n_atrib = con.execute('SELECT COUNT(*) FROM atribuicao_aval').fetchone()[0]
n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]

print(f'Aplicação: {n_app:,}')
print(f'Atribuídos: {n_atrib:,} ({100.0 * n_atrib / n_app:.1f}%)')
print(f'Sem CPF: {n_app - n_atrib:,} ({100.0 * (n_app - n_atrib) / n_app:.1f}%)')
print(f'Pares no parquet: {n_pred:,}')
print('Colunas faixa/regra:', tem_faixa, tem_regra)


Aplicação: 6,079,030
Atribuídos: 4,304,328 (70.8%)
Sem CPF: 1,774,702 (29.2%)
Pares no parquet: 6,315,410
Colunas faixa/regra: True True


## Notas e regra dos atribuídos

Distribuição de `match_probability`, contagem por `faixa`/`regra` do 04, e
Censos por CPF.


In [3]:
display(con.execute("""
SELECT
    CASE
        WHEN match_probability >= 0.99 THEN 'p_ge_099'
        WHEN match_probability >= 0.90 THEN 'p_090_099'
        WHEN match_probability >= 0.75 THEN 'p_075_090'
        WHEN match_probability >= 0.50 THEN 'p_050_075'
        WHEN match_probability >= 0.25 THEN 'p_025_050'
        ELSE 'p_lt_025'
    END AS faixa_p,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao_aval
GROUP BY 1
ORDER BY 1 DESC
""").df())

display(con.execute("""
SELECT
    faixa_atr,
    regra_atr,
    n,
    ROUND(100.0 * n / SUM(n) OVER (), 1) AS pct
FROM (
    SELECT
        coalesce(faixa, '(sem)') AS faixa_atr,
        coalesce(regra, '(sem)') AS regra_atr,
        COUNT(*) AS n
    FROM atribuicao_aval
    GROUP BY 1, 2
)
ORDER BY
    CASE faixa_atr
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
        ELSE 9
    END,
    regra_atr
""").df())

display(con.execute("""
SELECT
    n_censos,
    COUNT(*) AS n_cpf,
    n_censos * COUNT(*) AS n_pares
FROM (
    SELECT unique_id_cpf, COUNT(*) AS n_censos
    FROM atribuicao_aval
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
""").df())


,faixa_p,n,pct
0,p_ge_099,3832798,89.0
1,p_090_099,297222,6.9
2,p_075_090,45766,1.1
3,p_050_075,57525,1.3
4,p_025_050,71017,1.6


,faixa_atr,regra_atr,n,pct
0,modelo,score,4130020,96.0
1,resgate,R1,62354,1.4
2,resgate,R2,20897,0.5
3,resgate,R3,20040,0.5
4,baixo,R1,62882,1.5
5,baixo,R2,8135,0.2


,n_censos,n_cpf,n_pares
0,1,4081752,4081752
1,2,108573,217146
2,3,1810,5430


## Qualidade dos pares atribuídos

No par que saiu: mãe, endereço e distância de ano de nascimento.


In [4]:
display(con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
    ) / COUNT(*), 1) AS pct_mae_preenchida,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND ca.nome_mae_phon = pb.nome_mae_phon
    ) / COUNT(*), 1) AS pct_mae_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
          AND ca.logradouro_norm = pb.logradouro_norm
    ) / COUNT(*), 1) AS pct_logradouro_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
    ) / COUNT(*), 1) AS pct_cep_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
    ) / COUNT(*), 1) AS pct_data_igual
FROM atribuicao_aval a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
''').df())

display(con.execute(f'''
SELECT
    CASE
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'sem_data'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) = 0 THEN 'delta_0'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 1 THEN 'delta_1'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 5 THEN 'delta_2_5'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 10 THEN 'delta_6_10'
        ELSE 'delta_gt_10'
    END AS faixa_delta_ano,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao_aval a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
GROUP BY 1
ORDER BY 1
''').df())


,n,pct_mae_preenchida,pct_mae_igual,pct_logradouro_igual,pct_cep_igual,pct_data_igual
0,4304328,41.6,24.1,25.2,65.7,83.1


,faixa_delta_ano,n,pct
0,delta_0,3759455,87.3
1,delta_1,127662,3.0
2,delta_2_5,49583,1.2
3,delta_6_10,30031,0.7
4,delta_gt_10,11228,0.3
5,sem_data,326369,7.6


## Quem ficou de fora

Melhor par no parquet por Censo sem atribuição. Quem tinha `p ≥ 0,50` e
mesmo assim não saiu: teto ou falhou o resgate (R1/R2/R3).


In [5]:
# Mesmo desempate do 04: nota → mãe preenchida → logradouro → CEP.
con.execute(f'''
CREATE OR REPLACE TABLE melhor_qualquer AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        CAST(
            ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
            AS INTEGER
        ) DESC,
        CAST(
            ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
            AND ca.logradouro_norm = pb.logradouro_norm
            AS INTEGER
        ) DESC,
        CAST(
            ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
            AS INTEGER
        ) DESC,
        p.unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_piso AS
SELECT *
FROM melhor_qualquer
WHERE match_probability >= {PISO_RESGATE}
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id AS unique_id_censo
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao_aval a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute(f'''
CREATE OR REPLACE TABLE sem_cpf_melhor AS
SELECT
    s.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    CASE
        WHEN m.match_probability IS NULL THEN 'sem_par'
        WHEN m.match_probability >= 0.50 THEN 'p_ge_050'
        WHEN m.match_probability >= 0.25 THEN 'p_025_050'
        WHEN m.match_probability >= 0.05 THEN 'p_005_025'
        ELSE 'p_lt_005'
    END AS faixa_p
FROM censo_sem_cpf s
LEFT JOIN melhor_qualquer m ON m.unique_id_censo = s.unique_id_censo
''')

n_sem = con.execute('SELECT COUNT(*) FROM censo_sem_cpf').fetchone()[0]
assert n_atrib + n_sem == n_app

display(con.execute(f'''
SELECT
    a.faixa_p,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / {n_sem}, 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'p_ge_050'),
    (2, 'p_025_050'),
    (3, 'p_005_025'),
    (4, 'p_lt_005'),
    (5, 'sem_par')
) a(ord, faixa_p)
LEFT JOIN (
    SELECT faixa_p, COUNT(*) AS n
    FROM sem_cpf_melhor
    GROUP BY 1
) g ON g.faixa_p = a.faixa_p
ORDER BY a.ord
''').df())

# Censos com p >= PISO no melhor-par, mas CPF estourou o teto.
con.execute(f'''
CREATE OR REPLACE TABLE cpf_n_melhor_piso AS
SELECT unique_id_cpf, COUNT(*) AS n_censos
FROM melhor_piso
GROUP BY 1
''')

n_teto = con.execute(f'''
SELECT COUNT(*)
FROM sem_cpf_melhor s
JOIN cpf_n_melhor_piso c ON c.unique_id_cpf = s.unique_id_cpf
WHERE s.match_probability >= {PISO_RESGATE}
  AND c.n_censos > {TETO}
''').fetchone()[0]
n_alto = con.execute(f'''
SELECT COUNT(*) FROM sem_cpf_melhor
WHERE match_probability >= {PISO_RESGATE}
''').fetchone()[0]
print(f'Sem CPF com melhor p >= {PISO_RESGATE}: {n_alto:,}')
print(f'  dos quais teto (CPF com >{TETO} no melhor-par): {n_teto:,}')


,faixa_p,n,pct_sem_cpf,pct_aplicacao
0,p_ge_050,119281,6.7,2.0
1,p_025_050,132252,7.5,2.2
2,p_005_025,242856,13.7,4.0
3,p_lt_005,0,0.0,0.0
4,sem_par,1280313,72.1,21.1


Sem CPF com melhor p >= 0.5: 119,281
  dos quais teto (CPF com >3 no melhor-par): 386


## Pares aleatórios por faixa / regra

Amostra nova a cada execução (`ORDER BY random()`). Ajuste `N_AMOSTRA`.


In [6]:
N_AMOSTRA = 5

cols_par = '''
    a.match_probability,
    a.faixa,
    a.regra,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.nome_mae AS mae_censo,
    pb.nome_mae AS mae_cpf,
    ca.logradouro_norm AS logradouro_censo,
    pb.logradouro_norm AS logradouro_cpf,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf,
    a.unique_id_censo,
    a.unique_id_cpf
'''

print('--- Por faixa de nota (atribuídos) ---')
for nome, lo, hi in [
    ('p_ge_099', 0.99, 1.01),
    ('p_090_099', 0.90, 0.99),
    ('p_050_090', 0.50, 0.90),
    ('p_025_050', 0.25, 0.50),
]:
    print(f'\n=== {nome} [{lo}, {hi}) ===')
    display(con.execute(f'''
    SELECT {cols_par}
    FROM atribuicao_aval a
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
    WHERE a.match_probability >= {lo}
      AND a.match_probability < {hi}
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())

print('\n--- Por regra do 04 ---')
for regra in ['score', 'R1', 'R2', 'R3']:
    print(f'\n=== regra={regra} ===')
    display(con.execute(f'''
    SELECT {cols_par}
    FROM atribuicao_aval a
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
    WHERE a.regra = '{regra}'
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())

print('\n--- Sem CPF (melhor par), faixas abaixo do modelo ---')
for nome, lo, hi in [
    ('p_050_090', 0.50, 0.90),
    ('p_025_050', 0.25, 0.50),
    ('p_005_025', 0.05, 0.25),
]:
    print(f'\n=== {nome} [{lo}, {hi}) ===')
    display(con.execute(f'''
    SELECT
        s.match_probability,
        ca.nome_completo AS nome_censo,
        pb.nome_completo AS nome_cpf,
        ca.data_nascimento AS dob_censo,
        pb.data_nascimento AS dob_cpf,
        ca.nome_mae AS mae_censo,
        pb.nome_mae AS mae_cpf,
        ca.logradouro_norm AS logradouro_censo,
        pb.logradouro_norm AS logradouro_cpf,
        s.unique_id_censo,
        s.unique_id_cpf
    FROM sem_cpf_melhor s
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = s.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = s.unique_id_cpf
    WHERE s.match_probability >= {lo}
      AND s.match_probability < {hi}
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())


--- Por faixa de nota (atribuídos) ---

=== p_ge_099 [0.99, 1.01) ===


,match_probability,faixa,regra,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,cep_censo,cep_cpf,unique_id_censo,unique_id_cpf
0,1.000000,modelo,score,MAYELEN FERNANDA TORRES FIGUEIREDO,MAYELLEN FERNANDA TORRES FIGUEREDO,1996-08-08,1996-08-08,FRANCIANE TORRES FIGUEIREDO,FRANCIANE SILVA TORRES,DO EGITO,DO EGITO,65143000,65143000,censo_2101251050000020180020000326000001000000003,cpf_60834995344
1,1.000000,modelo,score,DANIEL SILVA NUNES,DANIEL SILVA NUNES,2019-02-05,2019-02-05,ANDREIA SABINA SILVA,ANDREIA SABINO SILVA,RAPOSA,CAMPESTRE,65925000,65925000,censo_2111805050000170001470000422000001000000005,cpf_10343021323
2,1.000000,modelo,score,ELVES SOUSA SILVA,ELVIS SOUSA SILVA,1993-02-03,1993-02-03,ANTONIA MARIA SOUSA,ANTONIA MARIA SOUSA,DOIS,02,65950000,65950000,censo_2101608050002630020010000082000001000000002,cpf_06351546338
3,1.000000,modelo,score,LUNNA RECECCA BARBOSA SOUSA,LUNNA REBECCA BARBOSA SOUSA,2022-04-13,2022-04-13,KELCILANE BATISTA BABORSA SOUSA,KELCILANE BATISTA BARBOSA SOUSA,VINTE E DOIS,MARAVILHA,65800000,65800000,censo_2101400050001040020030000018000001000000007,cpf_12373267373
4,0.995927,modelo,score,ANDERSON MENDES SANTOS,ANDERSON MENDES SANTOS,1997-07-05,1997-07-05,None,CONCITA MALHEIRO MENDES,AGRICULTORES,PRINCIPAL,None,65000000,censo_2111300050006800150020000358000001000000001,cpf_62007691337



=== p_090_099 [0.9, 0.99) ===


,match_probability,faixa,regra,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,cep_censo,cep_cpf,unique_id_censo,unique_id_cpf
0,0.955959,modelo,score,RAYSSA EVYLIN ANDRADE REGO,RAISSA EVELYN ANDRADE REGO,2003-11-21,2003-11-21,None,FRANCILEIDE MUNIZ ANDRADE,NOVE,CARLOS MACIEIRA,65041820,65000000,censo_2111300050002120050020000157000002000000004,cpf_61998667383
1,0.906695,modelo,score,FABIANO OLIVEIRA SOUSA,FABIANO SOUSA OLIVEIRA,None,1994-07-11,None,MARIA DORES SOUSA OLIVEIRA,PARAGUANORTE,PARAGUANORTE,65840000,65840000,censo_2111607050000380060030000226000001000000001,cpf_05205695390
2,0.976919,modelo,score,EMILLE SILVA SOUZA,EMILLY SILVA SOUSA,2005-11-18,2005-11-18,None,NARLANY PATRICIA AZEVEDO SILVA SOUSA,PIAUI,PRESIDENTE MEDICI,65036530,65763000,censo_2111300050003250010040000253000001000000002,cpf_61188452398
3,0.980835,modelo,score,FRACISCO AGUIAR SANTOS,FRANCISCO AGUIAR SANTOS,1956-10-22,1956-10-22,None,MARIA AGUIAR SANTOS,GARRASTAZU MEDICE,GARRASTASULK MEDECI,65110000,65110122,censo_2111201050001220030110000037000001000000001,cpf_18022952320
4,0.981455,modelo,score,RAIMUNDO NONATO COELHO,RAIMUNDO NONATO COELHO,1985-01-16,1985-01-16,None,MARIA JOANA CANTANHEDE COELHO,PUCAO,PAULICA,None,65430000,censo_2109304050000490001260000130000002000000001,cpf_00623128390



=== p_050_090 [0.5, 0.9) ===


...

,match_probability,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,unique_id_censo,unique_id_cpf
0,0.782260,GILSON JANSEN MOTA,GILSON JANCE MOTA,1987-07-29,1987-07-29,None,CECILIA JANCE MOTA,RIO GRANDE DO NORTE,PRINCIPAL,censo_2105302050002510070040000023000001000000002,cpf_60271435330
1,0.687139,ZIRAN LUCAS PEREIRA CALADO RABELO,ZINRA LUCAS PEREIRA CALADO RABELO,2005-06-05,2005-06-05,None,ANA PAULA PEREIRA CALADO,DOS GIRASSOIS,GIRASSOL,censo_2111201100000540170020000298000001000000002,cpf_62204898376
2,0.638161,ANTONIO ELISMAR SILVA ALVES,ANTONIO ELISMAR SILVA ALVES,2002-02-22,1976-11-18,ROSINELMA MOTA ANDRADE ALVES,LUCIMERE SILVA ALVES,BANDEIRANTES,CENTRO DO RAIMUNDAO,censo_2102358050000270130020000055000002000000003,cpf_03764533382
3,0.888899,LEANDRO ALMEIDA SILVA,LEANDRO ALMEIDA SILVA,None,1984-11-21,None,LUCILIA ALMEIDA SILVA,DA RODOVIARIA,TRES DE AGOSTO,censo_2104008050000550030080000145000002000000001,cpf_99488272368
4,0.534437,TIAGO ALVES PEREIRA,ANTONIO THIAGO ALVES PEREIRA,1994-09-01,1994-09-01,None,MARIA ALVES SANTOS,ADRIANO RODRIGUES,SANTOS DUMONT,censo_2111672050000020020030000043000001000000002,cpf_60694516350



=== p_025_050 [0.25, 0.5) ===


,match_probability,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,unique_id_censo,unique_id_cpf
0,0.289340,JOSE FRANCISCO RODRIGUES PAZ,JOSE FRANCISCO RODRIGUES PAZ,None,1972-03-18,None,FRANCISCA RODRIGUES PAZ,NO FELIX,MARCOS ROCHA,censo_2103307150000120001360000079000002000000002,cpf_78767164315
1,0.397675,SIMARA SOUSA ROCHA,TAMARA SILVA ROCHA,1994-10-24,1994-10-24,None,CLERIDE CALDAS SILVA,RAIMUNDO MORAIS,RODOVIARIA,censo_2104800050002260100030000078000001000000001,cpf_06175039319
2,0.390011,CICERO CONCEICAO,CICERO CONCEICAO,1946-07-03,1969-11-24,None,JOSEFA ANTONIA CONCEICAO,DO SOL,SANTO ANTONIO,censo_2102077050000210410010000299000001000000002,cpf_24570303234
3,0.340363,JOANA ABREL PENHA,JOANA ABREU PENHA,1957-04-04,1957-04-15,None,ALDERICE JOANA ABREU,SAO FELIPE,PV SAO FELIPE,censo_2106508050000290001280000267000001000000001,cpf_40499740378
4,0.377959,JOSE SANTOS CANTANHEIDE,JOSE SANTOS CANTANHEDE GOMES,1955-10-31,1955-10-31,None,JOSEFA CANTANHEDE GOMES,CARLINDO CAMARA LIMA,CARLINDO DO CAMARA LIMA,censo_2109601050000060020020000200000001000000002,cpf_10948180382



=== p_005_025 [0.05, 0.25) ===


,match_probability,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,unique_id_censo,unique_id_cpf
0,0.077442,MARIA LURDES FRAZAO,MARIA LURDES FRAZAO,1963-02-11,1944-07-04,None,FLORINDA LURDES FRAZAO,TRES,DO MATADOURO,censo_2111201100001290030030000145000001000000001,cpf_61069572314
1,0.143752,FLAVIO AUGUSTO SANTOS,FLAVIO AUGUSTO SANTOS,1988-05-11,1979-10-06,None,RAIMUNDA DOMINGAS SANTOS,BARRIGUDA,DA PETROBRAZ,censo_2102705050000570000030000313000002000000001,cpf_01300154314
2,0.068743,JISE JIRGE SOUSA,JOSE JORGE SOUSA,1969-04-18,1971-04-18,None,RITA SOUSA,NOVO MUNDO,PRINCIPAL,censo_2102606050000220100020000150000002000000004,cpf_63076197320
3,0.170281,HILSON CESAR SOUSA,HILSON SERGIO SOUSA LIMA,1981-01-15,1981-01-15,None,DOMINGAS SOUSA LIMA,01 DE MAIO,1 DE MAIO,censo_2105153050000140030010000108000001000000002,cpf_95948805387
4,0.215010,ANTONIA SILVA REIS,ANTONIA PEREIRA SILVA,1980-08-03,1980-08-03,None,FRANCISCA PEREIRA SILVA,PRINCIPAL,PRINCIPAL,censo_2102150050000030040010000106000001000000001,cpf_60449890384


## Observação sobre piso

O 04 já resgata `0,25 ≤ p < 0,90` com R1/R2/R3. Baixar o piso do **modelo**
(`p ≥ 0,90`) sem regra tende a puxar falso positivo — use a amostra acima
antes de afrouxar.


In [7]:
display(con.execute(f"""
SELECT
    faixa_atr,
    n,
    ROUND(100.0 * n / SUM(n) OVER (), 1) AS pct_atrib,
    ROUND(100.0 * n / {n_app}, 1) AS pct_aplicacao
FROM (
    SELECT
        coalesce(faixa, '(sem)') AS faixa_atr,
        COUNT(*) AS n
    FROM atribuicao_aval
    GROUP BY 1
)
ORDER BY
    CASE faixa_atr
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
        ELSE 9
    END
""").df())


,faixa_atr,n,pct_atrib,pct_aplicacao
0,modelo,4130020,96.0,67.9
1,resgate,103291,2.4,1.7
2,baixo,71017,1.6,1.2


## Lista ouro (censo_limpo)

Universo com a lista ainda dentro. Quem está na ouro e no pareamento conta
como `lista_ouro`. `nos_dois` tem de ser zero (essa gente saiu da aplicação
antes do 04).


In [8]:
ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{LISTA_OURO_ARQUIVO}') LIMIT 0"
    ).fetchall()
}
id_ouro, cpf_ouro = lista_ouro_colunas(ouro_cols)

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT
    CAST({id_ouro} AS VARCHAR) AS id_censo,
    {cpf_norm_sql(cpf_ouro)} AS cpf_norm
FROM read_parquet('{LISTA_OURO_ARQUIVO}')
WHERE {id_ouro} IS NOT NULL AND {cpf_ouro} IS NOT NULL
''')

display(con.execute(f'''
SELECT
    a.origem,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / nullif(t.n_censo, 0), 1) AS pct
FROM (VALUES
    (1, 'lista_ouro'),
    (2, 'pareamento'),
    (3, 'sem_cpf')
) a(ord, origem)
CROSS JOIN (
    SELECT COUNT(*) AS n_censo FROM {TABELA_CENSO_LIMPA}
) t
LEFT JOIN (
    SELECT
        CASE
            WHEN o.id_censo IS NOT NULL THEN 'lista_ouro'
            WHEN p.unique_id_censo IS NOT NULL THEN 'pareamento'
            ELSE 'sem_cpf'
        END AS origem,
        COUNT(*) AS n
    FROM {TABELA_CENSO_LIMPA} c
    LEFT JOIN (
        SELECT DISTINCT id_censo FROM lista_ouro
    ) o ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
    LEFT JOIN (
        SELECT DISTINCT unique_id_censo FROM atribuicao_aval
    ) p ON p.unique_id_censo = c.unique_id
    GROUP BY 1
) g ON g.origem = a.origem
ORDER BY a.ord
''').df())

n_ambos = con.execute(f'''
SELECT COUNT(*)
FROM {TABELA_CENSO_LIMPA} c
JOIN (SELECT DISTINCT id_censo FROM lista_ouro) o
  ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
JOIN (SELECT DISTINCT unique_id_censo FROM atribuicao_aval) p
  ON p.unique_id_censo = c.unique_id
''').fetchone()[0]
print('nos_dois:', f'{n_ambos:,}')
assert n_ambos == 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,origem,n,pct
0,lista_ouro,510774,7.8
1,pareamento,4304328,65.3
2,sem_cpf,1774702,26.9


nos_dois: 0


In [9]:
con.close()
print('Fim do 05.')


Fim do 05.
